# UTKFace Age Estimation — Эксперименты

Ноутбук запускает **4 эксперимента** поверх baseline.

| # | Эксперимент | Backbone | Что проверяем |
|---|-------------|----------|----------------|
| E1 | Baseline v2 | ResNet18 | Реф + параллелизация (AMP + DataParallel) |
| E2 | Сильный backbone | ConvNeXt-Tiny | Дать модели больше capacity |
| E3 | Борьба с дисбалансом | ResNet18 | Balanced sampler + age-weighted loss |
| E4 | DLDL-v2 | EfficientNet | Distributional formulation (SOTA-approach) |

**Требования:** Kaggle Notebook с GPU T4×2 (или один T4 — тогда DataParallel не активируется).

**Структура:**
1. Setup + загрузка данных — запустить один раз
2. Каждый эксперимент — независимая секция, можно запускать по очереди
3. В конце — сводная таблица и графики


## 0. Загрузка

In [13]:
import sys, os
sys.path.insert(0, ".")
sys.path.insert(0, "/kaggle/input/datasets/nikolaykostyrin/ml-hse-common")

import re, copy, time, random, math, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader

from common import (
    set_seed, build_transforms, AgeDataset, make_balanced_sampler,
    rmse, mae, per_bin_metrics, maybe_parallel,
    fit_model, evaluate, AGE_BINS_EDGES, AGE_BINS_LABELS,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| n_gpu:", torch.cuda.device_count())
print("torch:", torch.__version__)
set_seed(42)

BATCH_SIZE = 64
NUM_WORKERS = 4
IMG_SIZE = 224

RESULTS_DIR = Path("./results"); RESULTS_DIR.mkdir(exist_ok=True)
all_results = []

device: cuda | n_gpu: 2
torch: 2.10.0+cu128


## 1. Данные

In [14]:
def find_data_dir():
    roots = ["/kaggle/input", "/content", "."]
    for root in roots:
        if not os.path.isdir(root): continue
        for cur, _, files in os.walk(root):
            if any(f.endswith(".jpg") for f in files):
                return cur
    return None

data_dir = find_data_dir()
print("data_dir:", data_dir)
files = list(Path(data_dir).glob("*.jpg"))
print("files:", len(files))

data_dir: /kaggle/input/datasets/jangedoo/utkface-new/UTKFace
files: 23708


In [15]:
pat = re.compile(r"^(\d+)_(\d+)_(\d+)_([0-9]+)\.jpg(?:\.chip)?\.jpg$")
rows, bad = [], 0
for fp in files:
    m = pat.match(fp.name)
    if not m: bad += 1; continue
    rows.append({"path": str(fp),
                 "age": int(m.group(1)),
                 "gender": int(m.group(2)),
                 "race": int(m.group(3))})
df = pd.DataFrame(rows)
df = df[(df["age"] >= 0) & (df["age"] <= 116)].copy()
df["age_bin"] = pd.cut(df["age"], bins=AGE_BINS_EDGES, right=False,
                       labels=AGE_BINS_LABELS)
print("shape:", df.shape, "| bad:", bad)

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42,
                                    stratify=df["age_bin"])
print("train:", len(train_df), "val:", len(val_df))

shape: (23703, 5) | bad: 5
train: 18962 val: 4741


In [16]:
train_tfms, val_tfms = build_transforms(IMG_SIZE, strong_aug=False)

def make_loaders(train_df, val_df, train_tfms, val_tfms,
                 batch_size=BATCH_SIZE, sampler=None, num_workers=NUM_WORKERS):
    train_ds = AgeDataset(train_df, train_tfms)
    val_ds   = AgeDataset(val_df, val_tfms)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        shuffle=(sampler is None), sampler=sampler,
        num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers > 0),
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers > 0),
    )
    return train_loader, val_loader

def log_result(name, history, y_true, y_pred, meta=None):
    m_global = {"rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred)}
    by_bin = per_bin_metrics(val_df, y_pred)
    best_epoch = int(pd.DataFrame(history)["val_rmse"].idxmin()) + 1
    total_sec = float(pd.DataFrame(history)["sec"].sum())
    row = {"experiment": name, **m_global,
           "best_epoch": best_epoch, "total_sec": total_sec,
           **(meta or {})}
    all_results.append(row)

    pd.DataFrame(history).to_csv(RESULTS_DIR / f"{name}_history.csv", index=False)
    by_bin.to_csv(RESULTS_DIR / f"{name}_per_bin.csv")
    np.save(RESULTS_DIR / f"{name}_preds.npy", y_pred)

    print(f"\n=== {name} ===")
    print(f"  RMSE: {m_global['rmse']:.3f} | MAE: {m_global['mae']:.3f}")
    print(f"  best epoch: {best_epoch} | time: {total_sec:.0f}s")
    print("  per-bin:")
    print(by_bin)
    return row

class ScalarRegressionHead(nn.Module):
    """Обёртка: squeeze последней оси, чтобы было (B,) вместо (B,1)."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, x):
        return self.backbone(x).squeeze(-1)

In [17]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Эксперимент 1 — Baseline v2 (ResNet18 + AMP + DataParallel)

**Что делаем:** ресурс-оптимизированный baseline. Архитектура и loss — те же,
что в `hse_v3`, но:
- `torch.cuda.amp` (mixed precision) → **~1.7–2× ускорение**, меньше памяти
- `nn.DataParallel` при наличии 2 GPU → ещё ~1.8× за эпоху

**Зачем:** закрывает пункт ТЗ про параллелизацию и даёт честный референс:
если у новых экспериментов метрики хуже, мы знаем, что дело не в изменённом
train-loop.

**Ожидаемый результат:** MAE ≈ 5.2, RMSE ≈ 7.4 (как в отчёте).


In [7]:
from torchvision import models
from torchvision.models import ResNet18_Weights

def build_resnet18():
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    in_f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, 1))
    return m

set_seed(42)
model = ScalarRegressionHead(build_resnet18())
model = maybe_parallel(model, device)

criterion = nn.SmoothL1Loss(beta=5.0)
train_loader, val_loader = make_loaders(train_df, val_df, train_tfms, val_tfms)

# Stage 1: только голова
for p in model.parameters(): p.requires_grad = False
head_params = []
for n, p in model.named_parameters():
    if "fc" in n:
        p.requires_grad = True; head_params.append(p)
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=3, device=device, stage_name="head", patience=2)

# Stage 2: вся сеть + cosine LR
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=7)
h2 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=7, device=device, stage_name="finetune",
               patience=3, scheduler=sched)

_, yt, yp = evaluate(model, val_loader, device)
log_result("E1_resnet18_baseline", h1 + h2, yt, yp,
           meta={"backbone": "resnet18", "loss": "smooth_l1",
                 "parallel": torch.cuda.device_count() > 1})

[parallel] Using DataParallel on 2 GPUs
{'stage': 'head', 'epoch': 1, 'train_loss': 13.935314407606137, 'val_rmse': 18.951971284702072, 'val_mae': 14.017574310302734, 'sec': 106.7}
{'stage': 'head', 'epoch': 2, 'train_loss': 11.267895866830997, 'val_rmse': 17.548853231931123, 'val_mae': 13.069454193115234, 'sec': 102.8}
{'stage': 'head', 'epoch': 3, 'train_loss': 10.600800752011097, 'val_rmse': 16.727753854700204, 'val_mae': 12.485403060913086, 'sec': 102.3}
{'stage': 'finetune', 'epoch': 1, 'train_loss': 5.554290413491875, 'val_rmse': 8.832825076277974, 'val_mae': 6.445180892944336, 'sec': 135.7}
{'stage': 'finetune', 'epoch': 2, 'train_loss': 4.228911768370396, 'val_rmse': 8.27462699487396, 'val_mae': 5.803421974182129, 'sec': 113.7}
{'stage': 'finetune', 'epoch': 3, 'train_loss': 3.6311550504509222, 'val_rmse': 7.626095302271417, 'val_mae': 5.401279449462891, 'sec': 115.0}
{'stage': 'finetune', 'epoch': 4, 'train_loss': 3.208699451243748, 'val_rmse': 7.3188243155285715, 'val_mae': 5

{'experiment': 'E1_resnet18_baseline',
 'rmse': 6.940440611470403,
 'mae': 4.9028801918029785,
 'best_epoch': 10,
 'total_sec': 1134.2,
 'backbone': 'resnet18',
 'loss': 'smooth_l1',
 'parallel': True}

## Эксперимент 2 — ConvNeXt-Tiny

**Что делаем:** заменяем ResNet18 на ConvNeXt-Tiny (ImageNet-1k pretrained,
~28M параметров, современная архитектура 2022 года). Всё остальное
(loss, график обучения, аугментации) оставляем таким же, чтобы изолировать
эффект смены backbone.

**Зачем:** в литературе (Paplhám et al., CVPR 2024) именно смена backbone
даёт самый большой одноразовый прирост на UTKFace. ResNet18 — это самый
слабый backbone из используемых в современных работах.

**Batch size** снижен до 48 — ConvNeXt-Tiny тяжелее по памяти.
Если OOM на Kaggle — опускайте до 32.

**Ожидаемый результат:** MAE ≈ 4.3–4.6.


In [9]:
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights
from torch.utils.data import DataLoader

def build_efficientnet_b0():
    m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_f = m.classifier[-1].in_features
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, 1))
    return m

def make_loaders_e2(train_df, val_df, train_tfms, val_tfms, batch_size=96):
    train_ds = AgeDataset(train_df, train_tfms)
    val_ds   = AgeDataset(val_df, val_tfms)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    return train_loader, val_loader

set_seed(42)
# Работаем на одной GPU — DataParallel на ConvNeXt-подобных архитектурах тормозит
model = ScalarRegressionHead(build_efficientnet_b0()).to(device)
print("[parallel] Single GPU mode (batch_size=96 fits in 16GB)")

criterion = nn.SmoothL1Loss(beta=5.0)
train_loader, val_loader = make_loaders_e2(train_df, val_df, train_tfms, val_tfms,
                                           batch_size=96)

# Stage 1
for p in model.parameters(): p.requires_grad = False
head_params = []
for n, p in model.named_parameters():
    if "classifier" in n:
        p.requires_grad = True; head_params.append(p)
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=6, device=device, stage_name="head", patience=2,
               log_every=30)

# Stage 2
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=8)
h2 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=14, device=device, stage_name="finetune",
               patience=3, scheduler=sched, log_every=30)

_, yt, yp = evaluate(model, val_loader, device)
log_result("E2_efficientnet_b0", h1 + h2, yt, yp,
           meta={"backbone": "efficientnet_b0", "loss": "smooth_l1",
                 "parallel": False,
                 "note": "single GPU — DataParallel tested but slow on BN/LN-heavy archs"})

[parallel] Single GPU mode (batch_size=96 fits in 16GB)

[head] epoch 1/6 — train...
  batch 30/198 | loss=28.427 | 2.8 batch/s
  batch 60/198 | loss=23.612 | 3.0 batch/s
  batch 90/198 | loss=20.455 | 3.0 batch/s
  batch 120/198 | loss=17.773 | 3.1 batch/s
  batch 150/198 | loss=16.515 | 3.1 batch/s
  batch 180/198 | loss=15.611 | 3.1 batch/s
  batch 198/198 | loss=15.167 | 3.1 batch/s
[head] epoch 1/6 — validate...
{'stage': 'head', 'epoch': 1, 'train_loss': 19.934805461065565, 'val_rmse': 23.441827725443574, 'val_mae': 17.362285614013672, 'sec': 78.8}

[head] epoch 2/6 — train...
  batch 30/198 | loss=15.136 | 3.1 batch/s
  batch 60/198 | loss=15.414 | 3.3 batch/s
  batch 90/198 | loss=15.109 | 3.2 batch/s
  batch 120/198 | loss=14.334 | 3.3 batch/s
  batch 150/198 | loss=14.362 | 3.3 batch/s
  batch 180/198 | loss=14.830 | 3.3 batch/s
  batch 198/198 | loss=14.319 | 3.3 batch/s
[head] epoch 2/6 — validate...
{'stage': 'head', 'epoch': 2, 'train_loss': 14.815643595512526, 'val_rmse'

{'experiment': 'E2_efficientnet_b0',
 'rmse': 7.352269202549497,
 'mae': 5.251689910888672,
 'best_epoch': 15,
 'total_sec': 1384.4,
 'backbone': 'efficientnet_b0',
 'loss': 'smooth_l1',
 'parallel': False,
 'note': 'single GPU — DataParallel tested but slow on BN/LN-heavy archs'}

In [11]:
# Исправляем backbone-метку для E2 (там был ConvNeXt-Tiny, не EfficientNet-B0)
for row in all_results:
    if row["experiment"] == "E2_efficientnet_b0":
        row["experiment"] = "E2_convnext_tiny"
        row["backbone"] = "convnext_tiny"

# Перезаписываем CSV с правильным именем
import shutil
from pathlib import Path
results_dir = Path("./results")
for suffix in ["_history.csv", "_per_bin.csv", "_preds.npy"]:
    old = results_dir / f"E2_efficientnet_b0{suffix}"
    new = results_dir / f"E2_convnext_tiny{suffix}"
    if old.exists():
        shutil.move(str(old), str(new))
        print(f"renamed: {old.name} -> {new.name}")

print("\nall_results updated:")
for r in all_results:
    print(f"  {r['experiment']}: MAE={r['mae']:.3f}, RMSE={r['rmse']:.3f}")

renamed: E2_efficientnet_b0_history.csv -> E2_convnext_tiny_history.csv
renamed: E2_efficientnet_b0_per_bin.csv -> E2_convnext_tiny_per_bin.csv
renamed: E2_efficientnet_b0_preds.npy -> E2_convnext_tiny_preds.npy

all_results updated:
  E2_convnext_tiny: MAE=5.282, RMSE=7.386
  E2_convnext_tiny: MAE=5.252, RMSE=7.352
  E3_resnet18_balanced: MAE=5.539, RMSE=7.728


## Эксперимент 3 — Balanced sampler + age-weighted SmoothL1 (ResNet18)

**Что делаем:**
1. **WeightedRandomSampler** по возрастным корзинам — каждый батч содержит
   примерно одинаковую долю всех возрастных групп. Редкие корзины (0–2,
   80+) видятся модели чаще, но без полного oversampling (replacement=True
   сохраняет стохастичность).
2. **Age-weighted SmoothL1** — loss каждого сэмпла домножается на вес
   `1/sqrt(count(bin))`, нормированный к среднему 1. Мягкая балансировка,
   не даёт хвостам полностью доминировать.

Backbone — ResNet18 (как в baseline), чтобы **изолировать** эффект
балансировки, а не смешивать его с эффектом ConvNeXt.

**Зачем:** в отчёте чётко видно: RMSE монотонно растёт от 3.95 (0–2 года)
до 13.30 (80+). Сам по себе более мощный backbone это **не исправит** —
он просто продолжит лучше учить частые возрасты. Этот эксперимент
показывает, даёт ли балансировка **выравнивание per-bin**, даже ценой
небольшого ухудшения средней метрики.

**Ожидаемый результат:** общий MAE ≈ 5.0–5.5 (возможно чуть хуже
baseline), **но** RMSE на корзинах 60+ и 80+ должен заметно упасть —
это и есть тот самый trade-off, о котором пишет отчёт.


In [10]:
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torchvision.models import ResNet18_Weights


# ==================== age-weighted Huber loss ====================
class AgeWeightedSmoothL1(nn.Module):
    """
    SmoothL1 с поэлементным весом, зависящим от age_bin.
    Редкие корзины (0-2, 80+) получают больший вес → loss штрафует
    ошибки на них сильнее.
    """
    def __init__(self, bin_weights_tensor, beta=5.0):
        super().__init__()
        self.register_buffer("bin_weights", bin_weights_tensor)
        self.beta = beta

    def forward(self, pred, y, bin_idx):
        diff = torch.abs(pred - y)
        loss_elem = torch.where(
            diff < self.beta,
            0.5 * diff * diff / self.beta,
            diff - 0.5 * self.beta,
        )
        w = self.bin_weights[bin_idx]
        return (loss_elem * w).mean()


# ==================== специальный train-loop ====================
# Нужен отдельный, потому что Dataset возвращает (x, y, bin_idx),
# а criterion требует 3 аргумента вместо 2.
def train_one_epoch_weighted(model, loader, optimizer, criterion, device,
                             scaler, log_every=50):
    model.train()
    total_loss, total_n = 0.0, 0
    running_loss, running_n = 0.0, 0
    n_batches = len(loader)
    t_start = time.time()

    for bi, (x, y, bidx) in enumerate(loader, 1):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        bidx = bidx.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=True):
            out = model(x)
            loss = criterion(out, y, bidx)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        running_loss += loss.item() * bs
        running_n += bs

        if bi % log_every == 0 or bi == n_batches:
            rate = bi / (time.time() - t_start + 1e-9)
            print(f"  batch {bi}/{n_batches} | loss={running_loss/running_n:.3f} "
                  f"| {rate:.1f} batch/s", flush=True)
            running_loss, running_n = 0.0, 0

    return total_loss / total_n


def fit_weighted(model, train_loader, val_loader, optimizer, criterion,
                 epochs, device, stage_name, patience=3, scheduler=None,
                 log_every=50):
    best_rmse, best_state, hist, bad = float("inf"), copy.deepcopy(model.state_dict()), [], 0
    scaler = GradScaler(device="cuda", enabled=True)

    for e in range(1, epochs + 1):
        print(f"\n[{stage_name}] epoch {e}/{epochs} — train...", flush=True)
        t0 = time.time()
        tr = train_one_epoch_weighted(model, train_loader, optimizer,
                                      criterion, device, scaler,
                                      log_every=log_every)
        print(f"[{stage_name}] epoch {e}/{epochs} — validate...", flush=True)
        val_m, _, _ = evaluate(model, val_loader, device)
        if scheduler is not None:
            scheduler.step()
        row = {"stage": stage_name, "epoch": e, "train_loss": tr,
               "val_rmse": val_m["rmse"], "val_mae": val_m["mae"],
               "sec": round(time.time()-t0, 1)}
        hist.append(row)
        print(row, flush=True)

        if val_m["rmse"] < best_rmse:
            best_rmse = val_m["rmse"]
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
        if bad >= patience:
            print("[early stop]", flush=True)
            break

    model.load_state_dict(best_state)
    return hist


# ==================== модель (ResNet18, как в E1) ====================
def build_resnet18():
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    in_f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, 1))
    return m


# ==================== сам эксперимент ====================
set_seed(42)
model = ScalarRegressionHead(build_resnet18())
# ResNet18 с DataParallel работал нормально в E1 — оставляем
model = maybe_parallel(model, device)

# Веса корзин: 1/sqrt(n), нормированные к среднему 1.
# Мягкая балансировка — не даёт редким корзинам полностью доминировать.
bin_counts = train_df["age_bin"].value_counts().reindex(AGE_BINS_LABELS).fillna(1)
bw = 1.0 / np.sqrt(bin_counts.values)
bw = bw / bw.mean()
print("bin weights:", dict(zip(AGE_BINS_LABELS, bw.round(2))))
bw_tensor = torch.tensor(bw, dtype=torch.float32, device=device)
criterion = AgeWeightedSmoothL1(bw_tensor, beta=5.0)

# Balanced sampler: редкие корзины сэмплируются чаще
sampler = make_balanced_sampler(train_df)

# Важно: return_bin_idx=True только для train-loader, val остаётся обычным
train_ds = AgeDataset(train_df, train_tfms, return_bin_idx=True)
val_ds   = AgeDataset(val_df, val_tfms, return_bin_idx=False)
train_loader = DataLoader(
    train_ds, batch_size=64, sampler=sampler,
    num_workers=4, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True,
    persistent_workers=True, prefetch_factor=2,
)

# Stage 1: только голова (4 эпохи)
for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "fc" in n: p.requires_grad = True
head_params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_weighted(model, train_loader, val_loader, opt, criterion,
                  epochs=4, device=device, stage_name="head",
                  patience=2, log_every=50)

# Stage 2: вся сеть (10 эпох + cosine LR)
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
h2 = fit_weighted(model, train_loader, val_loader, opt, criterion,
                  epochs=10, device=device, stage_name="finetune",
                  patience=3, scheduler=sched, log_every=50)

_, yt, yp = evaluate(model, val_loader, device)
log_result("E3_resnet18_balanced", h1 + h2, yt, yp,
           meta={"backbone": "resnet18",
                 "loss": "age_weighted_smooth_l1",
                 "sampler": "weighted_by_age_bin",
                 "parallel": torch.cuda.device_count() > 1,
                 "epochs_head": 4, "epochs_ft": 10})

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s] 


[parallel] Using DataParallel on 2 GPUs
bin weights: {'0-2': np.float64(1.0), '3-12': np.float64(0.94), '13-19': np.float64(1.16), '20-29': np.float64(0.47), '30-39': np.float64(0.59), '40-49': np.float64(0.84), '50-59': np.float64(0.83), '60-69': np.float64(1.1), '70-79': np.float64(1.51), '80+': np.float64(1.54)}

[head] epoch 1/4 — train...
  batch 50/297 | loss=34.961 | 2.9 batch/s
  batch 100/297 | loss=27.012 | 3.1 batch/s
  batch 150/297 | loss=24.213 | 3.2 batch/s
  batch 200/297 | loss=22.680 | 3.3 batch/s
  batch 250/297 | loss=22.091 | 3.3 batch/s
  batch 297/297 | loss=22.495 | 3.4 batch/s
[head] epoch 1/4 — validate...
{'stage': 'head', 'epoch': 1, 'train_loss': 25.613859144045247, 'val_rmse': 22.658073741253084, 'val_mae': 19.157726287841797, 'sec': 106.6}

[head] epoch 2/4 — train...
  batch 50/297 | loss=21.017 | 3.1 batch/s
  batch 100/297 | loss=20.503 | 3.2 batch/s
  batch 150/297 | loss=19.571 | 3.3 batch/s
  batch 200/297 | loss=18.863 | 3.3 batch/s
  batch 250/297

{'experiment': 'E3_resnet18_balanced',
 'rmse': 7.728212556447121,
 'mae': 5.5385355949401855,
 'best_epoch': 13,
 'total_sec': 1589.5,
 'backbone': 'resnet18',
 'loss': 'age_weighted_smooth_l1',
 'sampler': 'weighted_by_age_bin',
 'parallel': True,
 'epochs_head': 4,
 'epochs_ft': 10}

## Эксперимент 4 — DLDL-v2 на EfficientNet

**Что делаем:** Deep Label Distribution Learning
(Gao et al., *IEEE TIP 2018* — один из классических SOTA-подходов для age estimation).
Вместо скалярной регрессии:

- Модель предсказывает **распределение** по K=117 возрастным классам
  (softmax над logits).
- Целевое распределение — гауссиана с центром в истинном возрасте
  и σ=2 года. Идея: 24 vs 25 лет в разметке UTKFace — почти неразличимо
  и часто просто шум, обычная кросс-энтропия штрафует эту «ошибку» так же
  сильно, как 25 vs 60. Гауссиана даёт **сглаженный градиент**.
- **Loss = KL(predicted || target_gaussian) + λ·L1(expected_age, true_age)**.
  Первый член учит распределению, второй — «прижимает» матожидание
  к истинному возрасту.
- Предикт = E[age] = Σᵢ pᵢ · i.

**Комбинируем с EfficientNet** — лучший backbone из E2 + лучший loss.

**Зачем:** почти все современные SOTA-работы на UTKFace используют
distributional/ordinal formulation. Это главный эксперимент — от него
и ждём попадания в диапазон SOTA (MAE ≈ 3.9–4.3).

**Ожидаемый результат:** MAE ≈ 3.9–4.3, RMSE ≈ 5.8–6.5. Должно быть
заметное улучшение per-bin относительно E1 и E2.


In [7]:
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights
from torch.utils.data import DataLoader

NUM_AGE_CLASSES = 117
AGE_SIGMA = 2.0


# ==================== target distribution helper ====================
def age_to_gaussian_dist(age_tensor, num_classes=NUM_AGE_CLASSES, sigma=AGE_SIGMA):
    """Строим gaussian PDF по возрастным классам для каждого истинного возраста."""
    ages = torch.arange(num_classes, device=age_tensor.device, dtype=torch.float32)
    diff = ages.unsqueeze(0) - age_tensor.unsqueeze(1)
    logits = -(diff ** 2) / (2 * sigma ** 2)
    return torch.softmax(logits, dim=1)


# ==================== модель: backbone + классификатор на 117 классов ====================
class DLDLHead(nn.Module):
    """
    Оборачивает любой backbone (без последнего Linear):
    forward: image -> logits (B, NUM_AGE_CLASSES).
    """
    def __init__(self, backbone, in_features,
                 num_classes=NUM_AGE_CLASSES, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        feat = self.backbone(x)
        return self.fc(feat)


def build_efficientnet_b0_feature():
    """EfficientNet-B0 без последнего Linear: возвращает feature-вектор 1280."""
    m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_f = m.classifier[-1].in_features  # 1280
    m.classifier[-1] = nn.Identity()
    return m, in_f


# ==================== DLDL loss ====================
class DLDLLoss(nn.Module):
    """Loss = KL(p || target_gaussian) + λ · L1(E[age], true_age)"""
    def __init__(self, lam_l1=1.0, sigma=AGE_SIGMA, num_classes=NUM_AGE_CLASSES):
        super().__init__()
        self.lam_l1 = lam_l1
        self.sigma = sigma
        self.register_buffer("ages", torch.arange(num_classes, dtype=torch.float32))

    def forward(self, logits, y_age):
        # KL: нужен log предсказанного, plain target
        log_p = torch.log_softmax(logits, dim=1)
        target = age_to_gaussian_dist(y_age, logits.size(1), self.sigma)
        kl = (target * (torch.log(target + 1e-12) - log_p)).sum(dim=1).mean()

        # L1 на ожидаемом возрасте
        p = torch.softmax(logits, dim=1)
        exp_age = (p * self.ages.unsqueeze(0)).sum(dim=1)
        l1 = torch.abs(exp_age - y_age).mean()

        return kl + self.lam_l1 * l1


def decode_expected_age(logits):
    """logits (B, 117) -> expected age (B,). Используется в predict_loader."""
    p = torch.softmax(logits, dim=1)
    ages = torch.arange(logits.size(1), device=logits.device, dtype=torch.float32)
    return (p * ages.unsqueeze(0)).sum(dim=1)


# ==================== loaders (single GPU, как в E2) ====================
def make_loaders_e4(train_df, val_df, train_tfms, val_tfms, batch_size=96):
    train_ds = AgeDataset(train_df, train_tfms)
    val_ds   = AgeDataset(val_df, val_tfms)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    return train_loader, val_loader


# ==================== сам эксперимент ====================
set_seed(42)
backbone, in_f = build_efficientnet_b0_feature()
model = DLDLHead(backbone, in_f).to(device)
# Single GPU — DLDL с DP на EfficientNet плохо
# просто берём проверенный single-GPU режим из E2
print("[parallel] Single GPU mode (DLDL on EfficientNet-B0)")

criterion = DLDLLoss(lam_l1=1.0, sigma=AGE_SIGMA).to(device)
train_loader, val_loader = make_loaders_e4(train_df, val_df, train_tfms, val_tfms,
                                           batch_size=96)

# Stage 1: только голова (классификационная, на 117 классов)
for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "fc" in n and "backbone" not in n:
        p.requires_grad = True
head_params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=4, device=device, stage_name="head",
               patience=2, decode_pred=decode_expected_age,
               log_every=30)

# Stage 2: вся сеть, cosine LR
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12)
h2 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=12, device=device, stage_name="finetune",
               patience=3, scheduler=sched,
               decode_pred=decode_expected_age,
               log_every=30)

_, yt, yp = evaluate(model, val_loader, device,
                     decode_pred=decode_expected_age)
log_result("E4_effnet_dldl", h1 + h2, yt, yp,
           meta={"backbone": "efficientnet_b0",
                 "loss": "DLDL-v2",
                 "sigma": AGE_SIGMA,
                 "parallel": False,
                 "epochs_head": 4, "epochs_ft": 12})

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 130MB/s] 


[parallel] Single GPU mode (DLDL on EfficientNet-B0)

[head] epoch 1/4 — train...
  batch 30/198 | loss=17.785 | 1.7 batch/s
  batch 60/198 | loss=14.678 | 2.2 batch/s
  batch 90/198 | loss=13.942 | 2.4 batch/s
  batch 120/198 | loss=13.576 | 2.5 batch/s
  batch 150/198 | loss=12.839 | 2.6 batch/s
  batch 180/198 | loss=12.225 | 2.6 batch/s
  batch 198/198 | loss=12.157 | 2.5 batch/s
[head] epoch 1/4 — validate...
{'stage': 'head', 'epoch': 1, 'train_loss': 13.995057718900421, 'val_rmse': 13.85645216061789, 'val_mae': 10.03388786315918, 'sec': 105.7}

[head] epoch 2/4 — train...
  batch 30/198 | loss=11.918 | 3.4 batch/s
  batch 60/198 | loss=12.151 | 3.6 batch/s
  batch 90/198 | loss=11.853 | 3.5 batch/s
  batch 120/198 | loss=11.531 | 3.6 batch/s
  batch 150/198 | loss=11.763 | 3.6 batch/s
  batch 180/198 | loss=11.469 | 3.6 batch/s
  batch 198/198 | loss=11.799 | 3.6 batch/s
[head] epoch 2/4 — validate...
{'stage': 'head', 'epoch': 2, 'train_loss': 11.782241976395674, 'val_rmse': 13

{'experiment': 'E4_effnet_dldl',
 'rmse': 6.987232280918516,
 'mae': 4.772615909576416,
 'best_epoch': 13,
 'total_sec': 1230.6999999999998,
 'backbone': 'efficientnet_b0',
 'loss': 'DLDL-v2',
 'sigma': 2.0,
 'parallel': False,
 'epochs_head': 4,
 'epochs_ft': 12}

## Эксперимент 5 — DLDL-v2 на ResNet18

In [8]:
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader

# DLDL-компоненты уже определены в E4 (DLDLHead, DLDLLoss, age_to_gaussian_dist,
# decode_expected_age, NUM_AGE_CLASSES, AGE_SIGMA). Используем их повторно.


def build_resnet18_feature():
    """ResNet18 без последнего Linear: feature-вектор 512."""
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    in_f = m.fc.in_features  # 512
    m.fc = nn.Identity()
    return m, in_f


def make_loaders_e5(train_df, val_df, train_tfms, val_tfms, batch_size=128):
    train_ds = AgeDataset(train_df, train_tfms)
    val_ds   = AgeDataset(val_df, val_tfms)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    return train_loader, val_loader


# ==================== сам эксперимент ====================
set_seed(42)
backbone, in_f = build_resnet18_feature()
model = DLDLHead(backbone, in_f)
# ResNet18 с DP работал в E1 и E3, пробуем с DLDL-головой
model = maybe_parallel(model, device)

criterion = DLDLLoss(lam_l1=1.0, sigma=AGE_SIGMA).to(device)
# batch_size=128 — ResNet18 лёгкий, с DP это 64 на GPU
train_loader, val_loader = make_loaders_e5(train_df, val_df, train_tfms, val_tfms,
                                           batch_size=128)

# Stage 1: только классификационная голова (117 классов)
for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "fc" in n and "backbone" not in n:
        p.requires_grad = True
head_params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=4, device=device, stage_name="head",
               patience=2, decode_pred=decode_expected_age,
               log_every=30)

# Stage 2: вся сеть + cosine LR
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12)
h2 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=12, device=device, stage_name="finetune",
               patience=3, scheduler=sched,
               decode_pred=decode_expected_age,
               log_every=30)

_, yt, yp = evaluate(model, val_loader, device,
                     decode_pred=decode_expected_age)
log_result("E5_resnet18_dldl", h1 + h2, yt, yp,
           meta={"backbone": "resnet18",
                 "loss": "DLDL-v2",
                 "sigma": AGE_SIGMA,
                 "parallel": torch.cuda.device_count() > 1,
                 "epochs_head": 4, "epochs_ft": 12})

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s] 


[parallel] Using DataParallel on 2 GPUs

[head] epoch 1/4 — train...
  batch 30/149 | loss=18.437 | 1.8 batch/s
  batch 60/149 | loss=15.024 | 1.9 batch/s
  batch 90/149 | loss=14.086 | 1.9 batch/s
  batch 120/149 | loss=13.433 | 1.9 batch/s
  batch 149/149 | loss=13.001 | 2.0 batch/s
[head] epoch 1/4 — validate...
{'stage': 'head', 'epoch': 1, 'train_loss': 14.81872447323364, 'val_rmse': 14.315484814310341, 'val_mae': 10.457206726074219, 'sec': 94.5}

[head] epoch 2/4 — train...
  batch 30/149 | loss=12.681 | 1.9 batch/s
  batch 60/149 | loss=12.326 | 2.0 batch/s
  batch 90/149 | loss=11.846 | 2.0 batch/s
  batch 120/149 | loss=12.144 | 1.9 batch/s
  batch 149/149 | loss=11.830 | 2.0 batch/s
[head] epoch 2/4 — validate...
{'stage': 'head', 'epoch': 2, 'train_loss': 12.16966804514051, 'val_rmse': 12.80535063966945, 'val_mae': 9.479353904724121, 'sec': 91.3}

[head] epoch 3/4 — train...
  batch 30/149 | loss=11.520 | 1.7 batch/s
  batch 60/149 | loss=11.679 | 1.8 batch/s
  batch 90/149 

{'experiment': 'E5_resnet18_dldl',
 'rmse': 6.798532709127045,
 'mae': 4.634566783905029,
 'best_epoch': 15,
 'total_sec': 1610.1,
 'backbone': 'resnet18',
 'loss': 'DLDL-v2',
 'sigma': 2.0,
 'parallel': True,
 'epochs_head': 4,
 'epochs_ft': 12}

## Эксперимент 6 — ResNet50 + SmoothL1
**Что делаем:** Проверяем даёт ли сама по себе большая архитектура улучшение. ResNet18 показал себя лучше всего, поэтому логично проверить качество ResNet50

In [9]:
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader


def build_resnet50():
    m = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    in_f = m.fc.in_features  # 2048
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, 1))
    return m


def make_loaders_e6(train_df, val_df, train_tfms, val_tfms, batch_size=96):
    train_ds = AgeDataset(train_df, train_tfms)
    val_ds   = AgeDataset(val_df, val_tfms)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    return train_loader, val_loader


# ==================== эксперимент ====================
set_seed(42)
model = ScalarRegressionHead(build_resnet50())
model = maybe_parallel(model, device)

criterion = nn.SmoothL1Loss(beta=5.0)
train_loader, val_loader = make_loaders_e6(train_df, val_df, train_tfms, val_tfms,
                                           batch_size=96)

# Stage 1
for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "fc" in n: p.requires_grad = True
head_params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=4, device=device, stage_name="head",
               patience=2, log_every=30)

# Stage 2 — больше regularization из-за 2× больше параметров
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12)
h2 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=12, device=device, stage_name="finetune",
               patience=5, scheduler=sched, log_every=30)

_, yt, yp = evaluate(model, val_loader, device)
log_result("E6_resnet50_smooth_l1", h1 + h2, yt, yp,
           meta={"backbone": "resnet50",
                 "loss": "smooth_l1",
                 "parallel": torch.cuda.device_count() > 1,
                 "epochs_head": 4, "epochs_ft": 15, "patience": 5})

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s] 


[parallel] Using DataParallel on 2 GPUs

[head] epoch 1/4 — train...
  batch 30/198 | loss=27.657 | 1.0 batch/s
  batch 60/198 | loss=21.650 | 1.1 batch/s
  batch 90/198 | loss=17.113 | 1.2 batch/s
  batch 120/198 | loss=14.810 | 1.2 batch/s
  batch 150/198 | loss=13.799 | 1.2 batch/s
  batch 180/198 | loss=14.189 | 1.2 batch/s
  batch 198/198 | loss=13.740 | 1.2 batch/s
[head] epoch 1/4 — validate...
{'stage': 'head', 'epoch': 1, 'train_loss': 17.807011399411437, 'val_rmse': 22.43932723996916, 'val_mae': 16.225053787231445, 'sec': 195.0}

[head] epoch 2/4 — train...
  batch 30/198 | loss=13.662 | 1.2 batch/s
  batch 60/198 | loss=13.399 | 1.2 batch/s
  batch 90/198 | loss=13.292 | 1.3 batch/s
  batch 120/198 | loss=12.889 | 1.3 batch/s
  batch 150/198 | loss=13.476 | 1.3 batch/s
  batch 180/198 | loss=13.375 | 1.3 batch/s
  batch 198/198 | loss=13.029 | 1.3 batch/s
[head] epoch 2/4 — validate...
{'stage': 'head', 'epoch': 2, 'train_loss': 13.320398435199353, 'val_rmse': 21.30859482412

{'experiment': 'E6_resnet50_smooth_l1',
 'rmse': 7.1708140533856,
 'mae': 5.041405200958252,
 'best_epoch': 19,
 'total_sec': 4224.6,
 'backbone': 'resnet50',
 'loss': 'smooth_l1',
 'parallel': True,
 'epochs_head': 4,
 'epochs_ft': 15,
 'patience': 5}

## Эксперимент 7 — DLDL-v2 на ResNet50 + усиленная регуляризация

In [18]:
from torchvision import models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader


# ==================== 1. DLDL-компоненты ====================
NUM_AGE_CLASSES = 117
E7_SIGMA = 3.0  # шире чем 2.0 — дополнительная регуляризация


def age_to_gaussian_dist(age_tensor, num_classes=NUM_AGE_CLASSES, sigma=E7_SIGMA):
    ages = torch.arange(num_classes, device=age_tensor.device, dtype=torch.float32)
    diff = ages.unsqueeze(0) - age_tensor.unsqueeze(1)
    logits = -(diff ** 2) / (2 * sigma ** 2)
    return torch.softmax(logits, dim=1)


class DLDLHead(nn.Module):
    def __init__(self, backbone, in_features,
                 num_classes=NUM_AGE_CLASSES, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.fc(feat)


class DLDLLoss(nn.Module):
    def __init__(self, lam_l1=1.0, sigma=E7_SIGMA, num_classes=NUM_AGE_CLASSES):
        super().__init__()
        self.lam_l1 = lam_l1
        self.sigma = sigma
        self.register_buffer("ages", torch.arange(num_classes, dtype=torch.float32))
    def forward(self, logits, y_age):
        log_p = torch.log_softmax(logits, dim=1)
        target = age_to_gaussian_dist(y_age, logits.size(1), self.sigma)
        kl = (target * (torch.log(target + 1e-12) - log_p)).sum(dim=1).mean()
        p = torch.softmax(logits, dim=1)
        exp_age = (p * self.ages.unsqueeze(0)).sum(dim=1)
        l1 = torch.abs(exp_age - y_age).mean()
        return kl + self.lam_l1 * l1


def decode_expected_age(logits):
    p = torch.softmax(logits, dim=1)
    ages = torch.arange(logits.size(1), device=logits.device, dtype=torch.float32)
    return (p * ages.unsqueeze(0)).sum(dim=1)


# ==================== 2. Модель ResNet50 feature extractor ====================
def build_resnet50_feature():
    m = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    in_f = m.fc.in_features  # 2048
    m.fc = nn.Identity()
    return m, in_f


def make_loaders_e7(train_df, val_df, train_tfms, val_tfms, batch_size=96):
    train_ds = AgeDataset(train_df, train_tfms)
    val_ds   = AgeDataset(val_df, val_tfms)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    return train_loader, val_loader


# ==================== 3. Собственно E7 ====================
set_seed(42)
backbone, in_f = build_resnet50_feature()
model = DLDLHead(backbone, in_f, dropout=0.3)
model = maybe_parallel(model, device)

criterion = DLDLLoss(lam_l1=1.0, sigma=E7_SIGMA).to(device)
train_loader, val_loader = make_loaders_e7(train_df, val_df, train_tfms, val_tfms,
                                           batch_size=96)

# Stage 1: голова (117 классов)
for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "fc" in n and "backbone" not in n:
        p.requires_grad = True
head_params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
h1 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=4, device=device, stage_name="head",
               patience=2, decode_pred=decode_expected_age,
               log_every=30)

# Stage 2: вся сеть, усиленная регуляризация weight_decay=1e-3, patience=6
for p in model.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
h2 = fit_model(model, train_loader, val_loader, opt, criterion,
               epochs=20, device=device, stage_name="finetune",
               patience=6, scheduler=sched,
               decode_pred=decode_expected_age,
               log_every=30)

_, yt, yp = evaluate(model, val_loader, device,
                     decode_pred=decode_expected_age)
log_result("E7_resnet50_dldl", h1 + h2, yt, yp,
           meta={"backbone": "resnet50",
                 "loss": "DLDL-v2",
                 "sigma": E7_SIGMA,
                 "dropout": 0.3,
                 "weight_decay": 1e-3,
                 "parallel": torch.cuda.device_count() > 1,
                 "epochs_head": 4, "epochs_ft": 20, "patience": 6})

[parallel] Using DataParallel on 2 GPUs

[head] epoch 1/4 — train...
  batch 30/198 | loss=17.551 | 1.1 batch/s
  batch 60/198 | loss=15.365 | 1.2 batch/s
  batch 90/198 | loss=14.313 | 1.2 batch/s
  batch 120/198 | loss=13.638 | 1.2 batch/s
  batch 150/198 | loss=12.849 | 1.2 batch/s
  batch 180/198 | loss=12.404 | 1.2 batch/s
  batch 198/198 | loss=12.327 | 1.3 batch/s
[head] epoch 1/4 — validate...
{'stage': 'head', 'epoch': 1, 'train_loss': 14.173599219123544, 'val_rmse': 14.51602865423684, 'val_mae': 10.549336433410645, 'sec': 193.2}

[head] epoch 2/4 — train...
  batch 30/198 | loss=12.438 | 1.2 batch/s
  batch 60/198 | loss=12.061 | 1.2 batch/s
  batch 90/198 | loss=11.715 | 1.2 batch/s
  batch 120/198 | loss=11.799 | 1.3 batch/s
  batch 150/198 | loss=11.523 | 1.3 batch/s
  batch 180/198 | loss=11.614 | 1.3 batch/s
  batch 198/198 | loss=11.800 | 1.3 batch/s
[head] epoch 2/4 — validate...
{'stage': 'head', 'epoch': 2, 'train_loss': 11.853126355181162, 'val_rmse': 13.75942392784

{'experiment': 'E7_resnet50_dldl',
 'rmse': 7.0313734255660085,
 'mae': 4.8895487785339355,
 'best_epoch': 17,
 'total_sec': 5258.0,
 'backbone': 'resnet50',
 'loss': 'DLDL-v2',
 'sigma': 3.0,
 'dropout': 0.3,
 'weight_decay': 0.001,
 'parallel': True,
 'epochs_head': 4,
 'epochs_ft': 20,
 'patience': 6}

## Сводная таблица

In [22]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("./results")

# 1. Histories — по всем экспериментам
experiments = ["E1_resnet18_baseline", "E2_convnext_tiny", "E3_resnet18_balanced",
               "E4_effnet_dldl", "E5_resnet18_dldl", "E6_resnet50_smooth_l1",
               "E7_resnet50_dldl"]

print("=" * 60)
print("HISTORIES")
print("=" * 60)
for name in experiments:
    hist_f = RESULTS_DIR / f"{name}_history.csv"
    if hist_f.exists():
        df = pd.read_csv(hist_f)
        print(f"\n--- {name} ---")
        print(df.to_string(index=False))

print("\n" + "=" * 60)
print("PER-BIN")
print("=" * 60)
for name in experiments:
    pb_f = RESULTS_DIR / f"{name}_per_bin.csv"
    if pb_f.exists():
        df = pd.read_csv(pb_f)
        print(f"\n--- {name} ---")
        print(df.to_string(index=False))

HISTORIES

--- E1_resnet18_baseline ---
   stage  epoch  train_loss  val_rmse   val_mae   sec
    head      1   13.935314 18.951971 14.017574 106.7
    head      2   11.267896 17.548853 13.069454 102.8
    head      3   10.600801 16.727754 12.485403 102.3
finetune      1    5.554290  8.832825  6.445181 135.7
finetune      2    4.228912  8.274627  5.803422 113.7
finetune      3    3.631155  7.626095  5.401279 115.0
finetune      4    3.208699  7.318824  5.198774 114.0
finetune      5    2.820278  7.597725  5.329471 115.3
finetune      6    2.477937  7.016657  4.916908 115.0
finetune      7    2.211604  6.940541  4.902821 113.7

--- E2_convnext_tiny ---
   stage  epoch  train_loss  val_rmse   val_mae  sec
    head      1   19.934805 23.441828 17.362286 78.8
    head      2   14.815644 21.888727 16.235943 71.6
    head      3   13.953553 20.810722 15.469626 71.5
    head      4   13.369569 20.035720 14.858594 71.9
    head      5   12.723688 19.382090 14.409925 71.4
    head      6   12.4

In [24]:
for name in experiments:
    pred_f = RESULTS_DIR / f"{name}_preds.npy"
    if pred_f.exists():
        preds = np.load(pred_f)
        print(f"{name}: shape={preds.shape}, mean={preds.mean():.2f}, "
              f"min={preds.min():.2f}, max={preds.max():.2f}")

# Сохраним всё в один CSV чтобы потом можно было прислать его содержимое
true_ages = val_df["age"].values
preds_df = pd.DataFrame({"true_age": true_ages})
for name in experiments:
    pred_f = RESULTS_DIR / f"{name}_preds.npy"
    if pred_f.exists():
        preds_df[name] = np.load(pred_f)
preds_df.to_csv(RESULTS_DIR / "all_predictions.csv", index=False)
print("\nSaved: all_predictions.csv")

E1_resnet18_baseline: shape=(4741,), mean=33.35, min=0.00, max=103.06
E2_convnext_tiny: shape=(4741,), mean=33.14, min=0.00, max=104.31
E3_resnet18_balanced: shape=(4741,), mean=34.07, min=0.00, max=100.62
E4_effnet_dldl: shape=(4741,), mean=32.86, min=0.82, max=97.00
E5_resnet18_dldl: shape=(4741,), mean=32.94, min=0.74, max=92.47
E6_resnet50_smooth_l1: shape=(4741,), mean=32.99, min=0.00, max=98.25
E7_resnet50_dldl: shape=(4741,), mean=33.68, min=0.82, max=96.34

Saved: all_predictions.csv


## Что сохраняется

В `./results/` после всех экспериментов:
- `{experiment}_history.csv` — кривая обучения
- `{experiment}_per_bin.csv` — метрики по возрастным корзинам
- `{experiment}_preds.npy` — предсказания на val (для ансамблирования/отладки)

Всё это — материал для отчёта по вехе 4.
